# Poke Agent Training

This notebook runs the training pipeline via the refactored `poke_agent` package.
It replaces the inlined code in `poke_agent_unified.ipynb` with importable modules.

- **Mac / local:** loads existing rollout JSONL and trains with Torch (CUDA, MPS, or CPU).
- **Linux / Kaggle:** can optionally generate a few CABT rollouts inline when `cg-lib` is available.
- **Does not submit** to the competition leaderboard.

Architecture docs: `docs/ARCHITECTURE.md` and `docs/poke-agent-modules.md`.

## 1. Setup

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import torch

# Ensure repo root is importable before loading poke_agent.
ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT.parent / "requirements.txt").exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from poke_agent.paths import print_runtime_info

print_runtime_info(ROOT)
print("torch", torch.__version__)

repo /home/inzi/poke-bot-agent
python 3.11.15
torch 2.12.1+cu130


## 2. Configuration

Defaults come from environment variables (see `docs/ARCHITECTURE.md`).
Override below before building config, or set env vars in the cell.

In [2]:
# Training requires CABT evaluation rollouts from the cg.game engine by default.
os.environ.setdefault("REQUIRE_CABT_EVAL_DATA", "1")

# Optional overrides — uncomment to change defaults for this session.
# os.environ["PRIMARY_ROLLOUT_DATA"] = "data/mac-rollouts-100k-fullstate.jsonl"
# os.environ["TRAIN_EPOCHS"] = "50"
# os.environ["BATCH_SIZE"] = "128"
# os.environ["MODEL_D_MODEL"] = "512"
# os.environ["CABT_EPISODES"] = "0"  # skip inline rollout generation
# os.environ["REQUIRE_CABT_EVAL_DATA"] = "0"  # smoke test only — allows synthetic fallback

from poke_agent.config import build_config

CONFIG = build_config(ROOT)
CONFIG

{'agent_deck_path': PosixPath('/home/inzi/poke-bot-agent/decks/competitive/high_performing/2026-05_regional-campinas-2026_4th_dragapult-dudunsparce.csv'),
 'data_candidates': [PosixPath('/home/inzi/poke-bot-agent/data/mac-rollouts-100k-fullstate.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/mac-rollouts-10k.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/notebook_rollouts.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/kaggle-output/data/cabt_rollouts.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/deckpool-smoke.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/container-mp-smoke.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/container-smoke.jsonl')],
 'generated_path': PosixPath('/home/inzi/poke-bot-agent/data/notebook_rollouts.jsonl'),
 'competition_results_path': PosixPath('/home/inzi/poke-bot-agent/data/competition-results.jsonl'),
 'output_path': PosixPath('/home/inzi/poke-bot-agent/out/value_model.pt'),
 'require_cabt_eval_data': True,
 'cabt_episo

## 3. Device and simulator

In [3]:
from poke_agent.device import torch_device
from poke_agent.simulator import load_simulator, print_simulator_status

DEVICE = torch_device()
print("device", DEVICE)

SIMULATOR = load_simulator(ROOT)
print_simulator_status(SIMULATOR)

device cuda
cg_lib_path /home/inzi/poke-bot-agent/kaggle/input/cg-lib
cg_available True


## 4. Deck and optional rollout generation

On Mac without `cg-lib`, rollout generation is skipped automatically.
For large datasets, use `scripts/generate_cabt_data.py` or Kaggle/Elmo workers instead.

In [4]:
from poke_agent.deck import read_deck
from poke_agent.rollout import generate_rollouts

DECK, DECK_SOURCE = read_deck(CONFIG, ROOT)
print("deck cards", len(DECK))
print("deck source", DECK_SOURCE)

GENERATE_EPISODES = int(os.environ.get("CABT_EPISODES", "3" if SIMULATOR.available else "0"))
generate_rollouts(SIMULATOR, DECK, GENERATE_EPISODES, CONFIG["generated_path"])

deck cards 60
deck source /home/inzi/poke-bot-agent/decks/competitive/high_performing/2026-05_regional-campinas-2026_4th_dragapult-dudunsparce.csv
generated 551 rows -> /home/inzi/poke-bot-agent/data/notebook_rollouts.jsonl


551

## 5. Load dataset and build tensors

Picks the first CABT evaluation JSONL from `CONFIG["data_candidates"]`.
Each row must include full `observation` / `action` / `next_observation` payloads
from `scripts/generate_cabt_data.py` (the same `cg.game` engine Kaggle uses).

Training **fails** if no valid CABT evaluation file is found unless
`REQUIRE_CABT_EVAL_DATA=0`.

In [5]:
from poke_agent.cabt_validation import assert_cabt_evaluation_rows, resolve_cabt_eval_data_path
from poke_agent.dataset import load_jsonl, prepare_training_tensors

DATA_PATH = resolve_cabt_eval_data_path(CONFIG["data_candidates"])
if DATA_PATH is None:
    raise RuntimeError(
        "No CABT evaluation rollout JSONL found. "
        "Run scripts/generate_cabt_data.py or set PRIMARY_ROLLOUT_DATA."
    )

PREVIEW_ROWS = load_jsonl(DATA_PATH)[:5]
assert_cabt_evaluation_rows(PREVIEW_ROWS, path=DATA_PATH, min_rows=1)
print("using CABT evaluation games from", DATA_PATH)

TENSORS = prepare_training_tensors(CONFIG, DEVICE)
print("feature dim", TENSORS.x.shape[1])
print("rows", TENSORS.x.shape[0])

jsonl load: 107582 rows using 30 workers
CABT evaluation data OK at /home/inzi/poke-bot-agent/data/mac-rollouts-100k-fullstate.jsonl: 5 rows, matchup=submission vs submission, feature_dim=10
using CABT evaluation games from /home/inzi/poke-bot-agent/data/mac-rollouts-100k-fullstate.jsonl
jsonl load: 107582 rows using 30 workers
CABT evaluation data OK at /home/inzi/poke-bot-agent/data/mac-rollouts-100k-fullstate.jsonl: 107582 rows, matchup=submission vs submission, feature_dim=10
tensor build: 107582 rows across 2200 episodes using 30 workers
loaded 107582 CABT evaluation rows from /home/inzi/poke-bot-agent/data/mac-rollouts-100k-fullstate.jsonl in 30.0s (30 workers)
x (107582, 266) value (107582,) transition (107582,)
history (107582, 1024) window 1024
feature dim 266
rows 107582


## 6. Build model and estimate VRAM

Builds the transformer and prints an estimated GPU memory budget before training starts.

In [6]:
from poke_agent.memory import print_vram_estimate
from poke_agent.training import build_model

MODEL = build_model(CONFIG, TENSORS, DEVICE)
print_vram_estimate(
    model=MODEL,
    param_count=sum(p.numel() for p in MODEL.parameters()),
    tensors=TENSORS,
    config=CONFIG,
    device=DEVICE,
)

model: d_model=256 heads=4 layers=4 ff=1024 dropout=0.1 window=1024
parameters: 3,824,148

VRAM estimate for current config
----------------------------------
device: cuda
data: 107,582 rows x 266 features (window=1024, batch=64)
parameters: 3,824,148
dataset tensors: 1.55 GiB
model weights:   14.59 MiB
gradients:       14.59 MiB
adam state:      29.18 MiB
batch peak:      15.82 GiB (attn 5.25 GiB forward, window² term dominates)
estimated total: 19.17 GiB (+ small PyTorch overhead)
gpu present:     11.62 GiB total, 9.09 GiB free before training


## 7. Train

Uses early stopping on total loss. Best weights are restored before checkpoint export.

In [ ]:
from poke_agent.training import train_model

TRAINING_REPORT = train_model(MODEL, TENSORS, CONFIG, DEVICE)
TRAINING_REPORT

training batches: rows=107582 batch_size=64 batches=1681


training:   0%|          | 0/1000 [00:00<?, ?epoch/s]

epoch 1/1000 batches:   0%|          | 0/1681 [00:00<?, ?batch/s]

## 8. Save checkpoint and report

In [ ]:
from poke_agent.checkpoint import print_training_report, save_checkpoint

OUTPUT_PATH = CONFIG["output_path"]
TRAINING_REPORT = save_checkpoint(
    model=MODEL,
    tensors=TENSORS,
    config=CONFIG,
    training_report=TRAINING_REPORT,
    output_path=OUTPUT_PATH,
)
print_training_report(TRAINING_REPORT, OUTPUT_PATH)

## 9. Inspect checkpoint (optional)

In [ ]:
import torch

checkpoint = torch.load(OUTPUT_PATH, map_location="cpu", weights_only=False)
{
    "model_type": checkpoint["model_type"],
    "input_dim": checkpoint["input_dim"],
    "policy_dim": checkpoint["policy_dim"],
    "model_config": checkpoint["model_config"],
    "best_total_loss": checkpoint["training_report"]["best_total_loss"],
    "best_epoch": checkpoint["training_report"]["best_epoch"],
    "data_path": checkpoint["data_path"],
}